In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy import *
from ipywidgets import interact, FloatSlider
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({'axes.labelsize': 18, 'xtick.labelsize': 16, 'ytick.labelsize': 16, 'axes.titlesize': 18,})
def logistic_map(r, x): # Function for Logistic
    return r * x * (1 - x)

def set_guess_r(true_r):
    if 0 <= true_r < 3:
        return 1
    elif 3 <= true_r <= 1 + np.sqrt(6):
        return true_r # Second period
    else:
        plt.figure(figsize=(4, 2))
        plt.text(0.5, 0.5, 'Error :( \n r too large', ha='center', va='center', fontsize=14, color='red') # Beyond r=3.44, give an ERROR
        plt.axis('off')
        plt.show()
        return None

sigma_Q = 0.5  # Controls variability of Q_k
def skf_logistic_r_estimation(r_true, r_guess, n_steps=100, Q=1e-5, Q_seq=0.5, x0_true=0.7, x0_hat=0.7, y_meas=None):
    if r_true < 3:
        n = 1
    elif r_true <= 1 + np.sqrt(6):
        n = 2
    else:
        n = 1
    Q0 = Q
    Q_seq = Q0 * np.exp(sigma_Q * np.random.randn(n_steps))
        
    if y_meas is None: # Real Map
        x_true = np.zeros(n_steps)
        y_meas = np.zeros(n_steps)
        x_true[0] = x0_true
        y_meas[0] = x_true[0] + np.random.normal(0, np.sqrt(Q_seq[0]))
        for k in range(1, n_steps):
            x_true[k] = logistic_map(r_true, x_true[k-1])
            y_meas[k] = x_true[k] + np.random.normal(0, np.sqrt(Q_seq[k]))
    else:
        x_true = None # If data does already exist, just use that

    x_hat = np.zeros(n_steps) # SKF Setup
    x_hat[0] = x0_hat
    P = np.eye(2)
    innovations = []

    if 0 <= r_true < 3:
        x_star = 1 - 1/r_guess
        Phi = np.array([[2 - r_guess, r_guess - 1], [0, 1]]) # 1-Periodic Region
    elif 3 <= r_true < 1 + np.sqrt(6):
        x_star = ((r_guess + 1) + np.sqrt((r_guess + 1)*(r_guess - 3))) / (2 * r_guess)
        Phi = np.array([[-r_guess**2 + 2*r_guess + 4, r_guess**2 - 2*r_guess - 3], [0, 1]]) # 2-Periodic Region

    for k in range(1, n_steps):
        C = np.array([[1, 0]])
        x_state = np.array([[x_hat[k-1]], [x_star]])

        x_pred = Phi @ x_state # Prediction stage
        P_pred = Phi @ P @ Phi.T
        
        y_pred = C @ x_pred # Update Measurement
        innovation = y_meas[k] - y_pred
        S = C @ P_pred @ C.T + Q_seq[k]
        K = P_pred @ C.T @ np.linalg.inv(S)

        x_update = x_pred + K * innovation
        P = (np.eye(2) - K @ C) @ P_pred @ (np.eye(2) - K @ C).T + K * Q_seq[k] * K.T # Joseph Form

        x_hat[k] = x_update[0, 0]
        innovations.append(float(innovation))
    return x_true, y_meas, x_hat, innovations

def interactive_skf(r_true=2.8, Q=1e-5):
    r_guess = set_guess_r(r_true)
    n_steps = 100
    n_runs = 100

    x_true_base = np.zeros(n_steps) # True Logistic Map (Single Trajectory)
    x_true_base[0] = 0.7
    for k in range(1, n_steps):
        x_true_base[k] = logistic_map(r_true, x_true_base[k-1])
    y_meas_base = x_true_base + np.random.normal(0, np.sqrt(Q), size=n_steps)

    all_x_true = np.zeros((n_runs, n_steps))
    all_x_hat = np.zeros((n_runs, n_steps))
    for i in range(n_runs): # Monte Carlo
        Q_seq = Q * np.exp(sigma_Q * np.random.randn(n_steps))
        y_meas = x_true_base + np.random.normal(0, np.sqrt(Q_seq), size=n_steps)
        x_true, y_meas, x_hat, innovations = skf_logistic_r_estimation(r_true = r_true, r_guess=r_guess, n_steps=n_steps, Q_seq=Q_seq)
        all_x_true[i, :] = x_true
        all_x_hat[i, :] = x_hat

    x_hat_mean = np.mean(all_x_hat, axis=0) # Statistics from MC
    x_hat_p05 = np.percentile(all_x_hat, 5, axis=0)
    x_hat_p95 = np.percentile(all_x_hat, 95, axis=0)
        
    mse_t = np.mean((all_x_hat - all_x_true)**2, axis=0) # MSE of MC
    variance_t = np.var(all_x_hat, axis=0) # Filter Spread (MC Variance)
        
    fig, ax = plt.subplots(1,3, figsize=(16,6), sharex=True) # Array Plot
    ax[0].plot(x_true, 'k-', lw=2, label='True Logistic Map')
    ax[0].plot(x_hat_mean, 'b--', lw=1.5, label='Mean SKF Estimate')
    ax[0].plot(y_meas_base, 'r.', alpha=0.4, label='Measurements')
    ax[0].fill_between(range(n_steps), x_hat_p05, x_hat_p95, color='b', alpha=0.2, label='95% Confidence Band')
    ax[0].set_title(f'SKF Monte Carlo (r_true={r_true:.3f}, Q={Q:.1e})')
    ax[0].set_xlabel('Iteration')
    ax[0].set_ylabel('x')
    ax[0].legend()

    ax[1].plot(mse_t, 'b-', label='MSE')
    ax[1].axhline(0, color='k', linestyle='--', linewidth=1)
    ax[1].set_xlabel('Iteration')
    ax[1].set_ylabel('MSE')
    ax[1].set_title('MSE Evolution Over Time')
    ax[1].legend()

    ax[2].plot(variance_t, 'r-', label='MC Variance')
    ax[2].axhline(0, color='k', linestyle='--', linewidth=1)
    ax[2].set_xlabel('Iteration')
    ax[2].set_ylabel('Variance')
    ax[2].set_title('Filter Spread of MC')
    ax[2].legend()
        
    plt.tight_layout()
    plt.show()        
    
interact( # Interactive Sliders
    interactive_skf,
    r_true=FloatSlider(value=2.8, min=2, max=4, step=0.01, description='True r'),
    Q=FloatSlider(value=1e-5, min=1e-15, max=0.01, step=1e-6, readout_format='.1e', description='Q'))

interactive(children=(FloatSlider(value=2.8, description='True r', max=4.0, min=2.0, step=0.01), FloatSlider(v…

<function __main__.interactive_skf(r_true=2.8, Q=1e-05)>